# Voz_Noslen F5-TTS para ONNX no Kaggle

Notebook pronto para baixar a voz F5-TTS do Hugging Face Buckets, copiar os arquivos essenciais para o worker do Kaggle, exportar um ONNX experimental do nucleo Transformer/DiT e enviar tudo para uma pasta nova em um Model Repo normal do Hugging Face.

Antes de rodar:

1. Ative internet no Kaggle.
2. Use GPU se disponivel.
3. Crie um Kaggle Secret chamado `HF_TOKEN` com permissao de leitura do bucket e escrita no repo de destino.
4. Execute as celulas em ordem.

Para caber no disco do Kaggle, o modo padrao `essential` baixa a voz escolhida, o checkpoint principal, o vocabulario, a referencia e configs pequenas. Ele ignora `.tmp` e checkpoints duplicados.


In [ ]:
# 1) Criar arquivos necessarios no worker do Kaggle
from pathlib import Path

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

requirements_text = 'huggingface_hub>=0.31.0,<1.0\nhf-xet>=1.1.0\nf5-tts>=1.1.9\nsafetensors>=0.4.5\ntorch>=2.4.0\ntorchaudio>=2.4.0\nsoundfile>=0.12.1\nlibrosa>=0.10.1,<0.11.0\npydub>=0.25.1\nvocos>=0.1.0\nhydra-core>=1.3.2\nomegaconf>=2.3.0\ncached-path>=1.5.1,<2.0.0\ntransformers>=4.36.0,<5.0.0\naccelerate>=0.25.0\ngradio>=4.44.0\nonnx>=1.16.0\nonnxscript>=0.1.0\nonnxruntime>=1.18.0\nnumpy\nscipy\npandas\n'
script_code = 'from __future__ import annotations\n\nimport argparse\nimport inspect\nimport json\nimport logging\nimport os\nimport shutil\nimport sys\nimport traceback\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import urljoin, urlparse, urlsplit, urlunsplit\n\n\nDEFAULT_SOURCE_URL = "https://huggingface.co/buckets/warllem/Voz_Noslen"\nDEFAULT_REVISION = "main"\nDEFAULT_VOICE_DIR = "voices/v_minha_voz_f5_tts_ptbr"\nPACKAGER_VERSION = "2026.06.15.6"\nWORK_ROOT = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))\nDOWNLOAD_DIR = WORK_ROOT / "voz_noslen_f5tts_snapshot"\nSTAGING_DIR = WORK_ROOT / "voz_noslen_onnx_package"\nLOG_PATH = WORK_ROOT / "voz_noslen_onnx_packager.log"\n\n\n@dataclass(frozen=True)\nclass PackagePaths:\n    source_snapshot: Path\n    staging_root: Path\n    copied_training_dir: Path\n    onnx_dir: Path\n    metadata_path: Path\n    export_report_path: Path\n\n\ndef setup_logging() -> logging.Logger:\n    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n    logger = logging.getLogger("voz_noslen_onnx_packager")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n\n    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n    file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(logging.Formatter("%(message)s"))\n\n    logger.addHandler(file_handler)\n    logger.addHandler(stream_handler)\n    return logger\n\n\nLOGGER = setup_logging()\n\n\ndef get_kaggle_secret(name: str) -> str | None:\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        value = UserSecretsClient().get_secret(name)\n        return value or None\n    except Exception:\n        return None\n\n\ndef get_hf_token() -> str | None:\n    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):\n        value = os.environ.get(name) or get_kaggle_secret(name)\n        if value:\n            return value\n    return None\n\n\ndef repo_id_from_url_or_id(value: str) -> str:\n    if not value.startswith(("http://", "https://")):\n        return value.strip("/")\n\n    parsed = urlparse(value)\n    parts = [part for part in parsed.path.strip("/").split("/") if part]\n    if parts[:1] in (["models"], ["model"]):\n        parts = parts[1:]\n    if parts[:1] in (["datasets"], ["spaces"]):\n        parts = parts[1:]\n    if parts[:1] == ["buckets"]:\n        parts = parts[1:]\n    if len(parts) < 2:\n        raise ValueError(f"Nao consegui resolver repo_id a partir de: {value}")\n    return "/".join(parts[:2])\n\n\ndef clean_dir(path: Path) -> None:\n    if path.exists():\n        shutil.rmtree(path)\n    path.mkdir(parents=True, exist_ok=True)\n\n\ndef copy_tree(src: Path, dst: Path) -> None:\n    if dst.exists():\n        shutil.rmtree(dst)\n    ignore = shutil.ignore_patterns(".git", ".cache", "__pycache__", "*.tmp")\n    shutil.copytree(src, dst, ignore=ignore)\n\n\ndef move_tree(src: Path, dst: Path) -> None:\n    if dst.exists():\n        shutil.rmtree(dst)\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    shutil.move(str(src), str(dst))\n\n\ndef find_first(root: Path, patterns: tuple[str, ...]) -> Path | None:\n    matches: list[Path] = []\n    for pattern in patterns:\n        matches.extend(root.glob(pattern))\n    files = sorted(path for path in matches if path.is_file())\n    return files[0] if files else None\n\n\ndef find_manifest(root: Path) -> Path | None:\n    preferred = sorted(root.glob("voices/*/manifest.json"))\n    if preferred:\n        return preferred[0]\n    return find_first(root, ("**/manifest.json",))\n\n\ndef find_checkpoint(root: Path, manifest: dict[str, Any] | None, manifest_path: Path | None) -> Path:\n    candidates: list[Path] = []\n    if manifest and manifest_path:\n        base_dir = manifest_path.parent\n        for key in ("voice_checkpoint", "inference_checkpoint", "final_checkpoint", "latest_checkpoint"):\n            value = manifest.get(key)\n            if not value:\n                continue\n            candidate = root / value if str(value).startswith(("voices/", "libraries/")) else base_dir / value\n            candidates.append(candidate)\n\n    candidates.extend(\n        sorted(root.glob(pattern))\n        for pattern in (\n            "**/model_*.pt",\n            "**/latest_checkpoint.pt",\n            "**/model_last.pt",\n            "**/model_last.safetensors",\n        )\n    )\n    flat_candidates: list[Path] = []\n    for item in candidates:\n        if isinstance(item, list):\n            flat_candidates.extend(item)\n        else:\n            flat_candidates.append(item)\n\n    existing = [path for path in flat_candidates if path.is_file()]\n    if not existing:\n        raise FileNotFoundError("Nenhum checkpoint .pt/.safetensors encontrado no pacote F5-TTS.")\n    return sorted(existing, key=lambda path: path.stat().st_size, reverse=True)[0]\n\n\ndef find_vocab(root: Path, checkpoint_path: Path) -> Path:\n    local = checkpoint_path.parent / "vocab.txt"\n    if local.is_file():\n        return local\n    vocab = find_first(root, ("voices/*/model/vocab.txt", "**/vocab.txt"))\n    if not vocab:\n        raise FileNotFoundError("Nenhum vocab.txt encontrado no pacote F5-TTS.")\n    return vocab\n\n\ndef find_reference_audio(root: Path, voice_dir: str) -> Path | None:\n    voice_ref = root / voice_dir / "data_reference" / "referencia_voz.wav"\n    if voice_ref.is_file():\n        return voice_ref\n    voice_refs = sorted(path for path in (root / voice_dir / "data_reference").glob("*.wav") if path.is_file())\n    if voice_refs:\n        return voice_refs[0]\n    return find_first(\n        root,\n        (\n            "voices/*/data_reference/referencia_voz.wav",\n            "voices/*/data_reference/*.wav",\n            "**/referencia*.wav",\n            "**/reference*.wav",\n            "**/*.wav",\n        ),\n    )\n\n\ndef load_json_if_exists(path: Path | None) -> dict[str, Any] | None:\n    if not path or not path.is_file():\n        return None\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef is_bucket_url(value: str) -> bool:\n    return value.startswith(("http://", "https://")) and "/buckets/" in urlparse(value).path\n\n\ndef strip_url_query(value: str) -> str:\n    parts = urlsplit(value)\n    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))\n\n\ndef bucket_relative_path(file_url: str) -> Path:\n    parsed = urlparse(strip_url_query(file_url))\n    parts = [part for part in parsed.path.split("/") if part]\n    for marker in ("resolve", "raw", "blob"):\n        if marker in parts:\n            index = parts.index(marker)\n            if len(parts) > index + 1:\n                remaining = parts[index + 1 :]\n                if remaining and remaining[0] in ("main", "refs", "resolve"):\n                    remaining = remaining[1:]\n                return Path(*remaining)\n    if "buckets" in parts and len(parts) > parts.index("buckets") + 3:\n        index = parts.index("buckets")\n        return Path(*parts[index + 3 :])\n    return Path(parts[-1])\n\n\ndef is_tmp_or_partial(path: Path) -> bool:\n    name = path.name.lower()\n    return name.endswith((".tmp", ".partial", ".incomplete"))\n\n\ndef choose_bucket_checkpoint(relative_paths: list[Path], voice_dir: str) -> Path | None:\n    voice_prefix = Path(voice_dir)\n    preferred_names = (\n        "model/model_2000.pt",\n        "model/latest_checkpoint.pt",\n        "model/model_last.pt",\n        "model/model_last.safetensors",\n        "model/final_checkpoint.pt",\n    )\n    available = {path.as_posix(): path for path in relative_paths}\n    for name in preferred_names:\n        candidate = (voice_prefix / name).as_posix()\n        if candidate in available:\n            return available[candidate]\n\n    checkpoints = [\n        path\n        for path in relative_paths\n        if path.as_posix().startswith(f"{voice_dir.rstrip(\'/\')}/model/")\n        and path.suffix.lower() in (".pt", ".safetensors")\n        and "base_checkpoint" not in path.name\n        and not is_tmp_or_partial(path)\n    ]\n    return sorted(checkpoints)[0] if checkpoints else None\n\n\ndef filter_bucket_files(file_urls: set[str], voice_dir: str, download_mode: str) -> list[tuple[str, Path]]:\n    entries = [(url, bucket_relative_path(url)) for url in file_urls]\n    entries = [(url, path) for url, path in entries if not is_tmp_or_partial(path)]\n    if download_mode == "all":\n        return sorted(entries, key=lambda item: item[1].as_posix())\n\n    relative_paths = [path for _, path in entries]\n    checkpoint = choose_bucket_checkpoint(relative_paths, voice_dir)\n    if checkpoint is None:\n        raise RuntimeError(f"Nenhum checkpoint principal encontrado em {voice_dir}/model.")\n\n    wanted: set[str] = {checkpoint.as_posix()}\n    voice_prefix = voice_dir.rstrip("/")\n    for _, path in entries:\n        path_text = path.as_posix()\n        if path_text == ".gitattributes":\n            wanted.add(path_text)\n        if path_text.startswith(f"{voice_prefix}/"):\n            if path_text.endswith((".md", ".json", ".txt", ".wav")):\n                wanted.add(path_text)\n            if path_text == f"{voice_prefix}/model/vocab.txt":\n                wanted.add(path_text)\n        if path_text.startswith("libraries/"):\n            if path_text.endswith((".md", ".json", ".txt", ".wav")):\n                wanted.add(path_text)\n\n    LOGGER.info("Modo essential: checkpoint escolhido para download: %s", checkpoint.as_posix())\n    LOGGER.info("Modo essential: %s arquivo(s) selecionado(s), checkpoints duplicados e .tmp ignorados.", len(wanted))\n    return sorted((url, path) for url, path in entries if path.as_posix() in wanted)\n\n\ndef download_http_file(url: str, output_path: Path, token: str | None) -> None:\n    import requests\n\n    headers = {"Authorization": f"Bearer {token}"} if token else {}\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with requests.get(url, headers=headers, stream=True, timeout=60) as response:\n        response.raise_for_status()\n        with output_path.open("wb") as handle:\n            for chunk in response.iter_content(chunk_size=1024 * 1024):\n                if chunk:\n                    handle.write(chunk)\n\n\ndef download_bucket_source(source_url: str, token: str | None, voice_dir: str, download_mode: str) -> Path:\n    import re\n    import requests\n\n    clean_dir(DOWNLOAD_DIR)\n    headers = {"Authorization": f"Bearer {token}"} if token else {}\n    source_url = source_url.rstrip("/")\n    pages = [source_url, f"{source_url}/tree/main"]\n    seen_pages: set[str] = set()\n    file_urls: set[str] = set()\n\n    LOGGER.info("Origem parece ser Hugging Face Buckets; tentando listar links HTML em %s", source_url)\n    while pages:\n        page_url = pages.pop(0)\n        if page_url in seen_pages:\n            continue\n        seen_pages.add(page_url)\n        response = requests.get(page_url, headers=headers, timeout=60)\n        if response.status_code == 404:\n            continue\n        response.raise_for_status()\n\n        for href in re.findall(r\'href=["\\\']([^"\\\']+)["\\\']\', response.text):\n            absolute = strip_url_query(urljoin(page_url, href))\n            parsed = urlparse(absolute)\n            if parsed.netloc != urlparse(source_url).netloc:\n                continue\n            if "/tree/" in parsed.path and "/buckets/" in parsed.path and absolute not in seen_pages:\n                pages.append(absolute)\n            if any(marker in parsed.path for marker in ("/resolve/", "/raw/", "/blob/")) and "/buckets/" in parsed.path:\n                file_urls.add(absolute.replace("/blob/", "/resolve/").replace("/raw/", "/resolve/"))\n\n    if not file_urls:\n        raise RuntimeError(\n            "Nao consegui listar arquivos do link /buckets/. Esse endereco nao e um Model Repo padrao do Hugging Face. "\n            "Abra o bucket no navegador, copie os arquivos para um Model Repo normal ou informe o repo_id correto em --repo-id. "\n            f"Origem recebida: {source_url}"\n        )\n\n    selected_files = filter_bucket_files(file_urls, voice_dir, download_mode)\n    for url, relative in selected_files:\n        output_path = DOWNLOAD_DIR / relative\n        LOGGER.info("Baixando bucket: %s -> %s", url, output_path)\n        download_http_file(url, output_path, token)\n\n    return DOWNLOAD_DIR\n\n\ndef download_source_repo(repo_id: str, revision: str, token: str | None) -> Path:\n    from huggingface_hub import snapshot_download\n\n    clean_dir(DOWNLOAD_DIR)\n    LOGGER.info("Baixando snapshot de %s @ %s para %s", repo_id, revision, DOWNLOAD_DIR)\n    try:\n        snapshot_download(\n            repo_id=repo_id,\n            repo_type="model",\n            revision=revision,\n            local_dir=str(DOWNLOAD_DIR),\n            token=token,\n            ignore_patterns=(".git/*",),\n        )\n    except TypeError:\n        snapshot_download(\n            repo_id=repo_id,\n            repo_type="model",\n            revision=revision,\n            local_dir=str(DOWNLOAD_DIR),\n            token=token,\n        )\n    return DOWNLOAD_DIR\n\n\ndef download_source(\n    source: str,\n    repo_id: str | None,\n    revision: str,\n    token: str | None,\n    voice_dir: str,\n    download_mode: str,\n) -> tuple[Path, str]:\n    if repo_id:\n        return download_source_repo(repo_id, revision, token), repo_id\n    if is_bucket_url(source):\n        return download_bucket_source(source, token, voice_dir, download_mode), source\n    resolved_repo_id = repo_id_from_url_or_id(source)\n    return download_source_repo(resolved_repo_id, revision, token), resolved_repo_id\n\n\ndef make_package_paths() -> PackagePaths:\n    clean_dir(STAGING_DIR)\n    copied_training_dir = STAGING_DIR / "f5_tts_original"\n    onnx_dir = STAGING_DIR / "onnx"\n    onnx_dir.mkdir(parents=True, exist_ok=True)\n    return PackagePaths(\n        source_snapshot=DOWNLOAD_DIR,\n        staging_root=STAGING_DIR,\n        copied_training_dir=copied_training_dir,\n        onnx_dir=onnx_dir,\n        metadata_path=STAGING_DIR / "package_metadata.json",\n        export_report_path=STAGING_DIR / "onnx_export_report.json",\n    )\n\n\ndef build_default_f5_config(manifest: dict[str, Any] | None) -> dict[str, Any]:\n    exp_name = (manifest or {}).get("exp_name") or "F5TTS_v1_Base"\n    if exp_name != "F5TTS_v1_Base":\n        raise RuntimeError(f"Exportador preparado apenas para F5TTS_v1_Base; encontrado: {exp_name!r}")\n    return {\n        "exp_name": exp_name,\n        "backbone": "DiT",\n        "arch": {\n            "dim": 1024,\n            "depth": 22,\n            "heads": 16,\n            "ff_mult": 2,\n            "text_dim": 512,\n            "text_mask_padding": True,\n            "qk_norm": None,\n            "conv_layers": 4,\n            "pe_attn_head": None,\n            "attn_backend": "torch",\n            "attn_mask_enabled": False,\n            "checkpoint_activations": False,\n        },\n        "mel_spec": {\n            "target_sample_rate": 24000,\n            "n_mel_channels": 100,\n            "hop_length": 256,\n            "win_length": 1024,\n            "n_fft": 1024,\n            "mel_spec_type": "vocos",\n        },\n        "tokenizer": (manifest or {}).get("tokenizer") or "char",\n    }\n\n\ndef infer_module_float_dtype(module: Any) -> Any:\n    import torch\n\n    for parameter in module.parameters():\n        if parameter.is_floating_point():\n            return parameter.dtype\n    for buffer in module.buffers():\n        if buffer.is_floating_point():\n            return buffer.dtype\n    return torch.float32\n\n\nclass F5TransformerOnnxWrapper:\n    def __init__(self, model: Any, compute_dtype: Any) -> None:\n        import torch\n\n        class Wrapper(torch.nn.Module):\n            def __init__(self, inner_model: Any, dtype: Any) -> None:\n                super().__init__()\n                self.inner_model = inner_model\n                self.transformer = getattr(inner_model, "transformer", inner_model)\n                self.compute_dtype = dtype\n\n            def _cast_float_input(self, value):\n                if torch.is_tensor(value) and value.is_floating_point() and value.dtype != self.compute_dtype:\n                    return value.to(dtype=self.compute_dtype)\n                return value\n\n            def forward(self, x, cond, text, time, mask):\n                x = self._cast_float_input(x)\n                cond = self._cast_float_input(cond)\n                time = self._cast_float_input(time)\n                try:\n                    return self.transformer(\n                        x=x,\n                        cond=cond,\n                        text=text,\n                        time=time,\n                        mask=mask,\n                        drop_audio_cond=False,\n                        drop_text=False,\n                    )\n                except TypeError:\n                    return self.transformer(\n                        x,\n                        cond,\n                        text,\n                        time,\n                        mask=mask,\n                        drop_audio_cond=False,\n                        drop_text=False,\n                    )\n\n        self.module = Wrapper(model, compute_dtype)\n\n\ndef kwargs_supported_by(function: Any, kwargs: dict[str, Any]) -> dict[str, Any]:\n    try:\n        signature = inspect.signature(function)\n    except (TypeError, ValueError):\n        return kwargs\n    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values()):\n        return kwargs\n    return {key: value for key, value in kwargs.items() if key in signature.parameters}\n\n\ndef legacy_onnx_export(\n    model: Any,\n    sample_inputs: tuple[Any, ...],\n    onnx_path: Path,\n    input_names: list[str],\n    output_names: list[str],\n) -> str:\n    import torch\n\n    common_kwargs = {\n        "input_names": input_names,\n        "output_names": output_names,\n        "opset_version": 18,\n        "do_constant_folding": True,\n    }\n\n    try:\n        from torch.onnx import utils as onnx_utils\n\n        LOGGER.info("Tentando exportacao ONNX pelo caminho legado torch.onnx.utils.export.")\n        legacy_kwargs = {\n            **common_kwargs,\n            "use_external_data_format": True,\n        }\n        onnx_utils.export(\n            model,\n            sample_inputs,\n            str(onnx_path),\n            **kwargs_supported_by(onnx_utils.export, legacy_kwargs),\n        )\n        return "torch.onnx.utils.export"\n    except Exception as exc:\n        LOGGER.warning("Falha no caminho legado direto: %s: %s", type(exc).__name__, exc)\n\n    try:\n        LOGGER.info("Tentando exportacao ONNX por torch.onnx.export com dynamo=False.")\n        export_kwargs = {\n            **common_kwargs,\n            "dynamo": False,\n            "external_data": True,\n        }\n        torch.onnx.export(\n            model,\n            sample_inputs,\n            str(onnx_path),\n            **kwargs_supported_by(torch.onnx.export, export_kwargs),\n        )\n        return "torch.onnx.export_dynamo_false"\n    except Exception as exc:\n        LOGGER.warning("Falha em torch.onnx.export com dynamo=False: %s: %s", type(exc).__name__, exc)\n\n    LOGGER.info("Tentando torch.jit.trace antes da exportacao ONNX.")\n    traced = torch.jit.trace(model, sample_inputs, strict=False)\n    traced.eval()\n    export_kwargs = {\n        **common_kwargs,\n        "dynamo": False,\n        "external_data": True,\n    }\n    torch.onnx.export(\n        traced,\n        sample_inputs,\n        str(onnx_path),\n        **kwargs_supported_by(torch.onnx.export, export_kwargs),\n    )\n    return "torch.jit.trace_then_torch.onnx.export"\n\n\ndef export_f5_core_to_onnx(checkpoint_path: Path, vocab_path: Path, onnx_dir: Path, manifest: dict[str, Any] | None) -> dict[str, Any]:\n    import torch\n    from f5_tts.infer.utils_infer import load_model\n    from hydra.utils import get_class\n\n    try:\n        import onnxscript  # noqa: F401\n    except ImportError as exc:\n        raise RuntimeError(\n            "Dependencia ausente: onnxscript. Rode novamente a celula 2 do notebook atualizado "\n            "ou execute: pip install onnxscript"\n        ) from exc\n\n    config = build_default_f5_config(manifest)\n    model_cls = get_class(f"f5_tts.model.{config[\'backbone\']}")\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    onnx_path = onnx_dir / "f5_tts_transformer_core.onnx"\n\n    LOGGER.info("Carregando F5-TTS em %s para exportacao ONNX", device)\n    try:\n        model = load_model(\n            model_cls,\n            config["arch"],\n            str(checkpoint_path),\n            mel_spec_type=config["mel_spec"]["mel_spec_type"],\n            vocab_file=str(vocab_path),\n            use_ema=True,\n            device=device,\n        )\n        use_ema = True\n    except Exception:\n        LOGGER.warning("Falha ao carregar com EMA; tentando carregar pesos sem EMA.")\n        model = load_model(\n            model_cls,\n            config["arch"],\n            str(checkpoint_path),\n            mel_spec_type=config["mel_spec"]["mel_spec_type"],\n            vocab_file=str(vocab_path),\n            use_ema=False,\n            device=device,\n        )\n        use_ema = False\n    model.eval()\n    model_compute_dtype = infer_module_float_dtype(model)\n    LOGGER.info("Dtype de ponto flutuante do modelo: %s", model_compute_dtype)\n    wrapped = F5TransformerOnnxWrapper(model, model_compute_dtype).module.to(device).eval()\n\n    batch = 1\n    frames = 64\n    text_tokens = 32\n    mel_channels = config["mel_spec"]["n_mel_channels"]\n    x = torch.randn(batch, frames, mel_channels, device=device)\n    cond = torch.zeros(batch, frames, mel_channels, device=device)\n    text = torch.randint(0, max(2, sum(1 for _ in vocab_path.open(encoding="utf-8"))), (batch, text_tokens), device=device)\n    time = torch.full((batch,), 0.5, dtype=torch.float32, device=device)\n    mask = torch.ones(batch, frames, dtype=torch.bool, device=device)\n\n    LOGGER.info("Exportando nucleo Transformer para %s", onnx_path)\n    export_method = legacy_onnx_export(\n        wrapped,\n        (x, cond, text, time, mask),\n        onnx_path,\n        ["x", "cond", "text", "time", "mask"],\n        ["pred"],\n    )\n\n    report: dict[str, Any] = {\n        "status": "ok",\n        "packager_version": PACKAGER_VERSION,\n        "onnx_file": str(onnx_path),\n        "checkpoint": str(checkpoint_path),\n        "vocab": str(vocab_path),\n        "device": device,\n        "use_ema": use_ema,\n        "model_compute_dtype": str(model_compute_dtype),\n        "export_mode": "legacy_static",\n        "export_method": export_method,\n        "static_shapes": {"batch": batch, "frames": frames, "text_tokens": text_tokens, "mel_channels": mel_channels},\n        "note": "Este ONNX contem o nucleo Transformer/DiT do F5-TTS com formas estaticas. O pacote mantem os arquivos originais para inferencia Python completa.",\n    }\n    validate_onnx(onnx_path, report)\n    return report\n\n\ndef validate_onnx(onnx_path: Path, report: dict[str, Any]) -> None:\n    import onnx\n\n    model = onnx.load(str(onnx_path))\n    onnx.checker.check_model(model)\n    report["onnx_checker"] = "ok"\n    try:\n        import onnxruntime as ort\n\n        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n        report["onnxruntime_inputs"] = [item.name for item in session.get_inputs()]\n        report["onnxruntime_outputs"] = [item.name for item in session.get_outputs()]\n        report["onnxruntime_load"] = "ok"\n    except Exception as exc:\n        report["onnxruntime_load"] = f"falhou: {type(exc).__name__}: {exc}"\n\n\ndef write_package_metadata(\n    paths: PackagePaths,\n    repo_id: str,\n    revision: str,\n    hf_folder: str,\n    checkpoint_path: Path,\n    vocab_path: Path,\n    reference_audio_path: Path | None,\n    manifest_path: Path | None,\n    manifest: dict[str, Any] | None,\n    export_report: dict[str, Any],\n) -> None:\n    metadata = {\n        "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "source_repo_id": repo_id,\n        "source_revision": revision,\n        "target_huggingface_folder": hf_folder,\n        "policy": "Arquivos originais copiados; nenhum arquivo do treinamento remoto e alterado.",\n        "copied_training_dir": str(paths.copied_training_dir),\n        "checkpoint": str(checkpoint_path.relative_to(paths.copied_training_dir)),\n        "vocab": str(vocab_path.relative_to(paths.copied_training_dir)),\n        "reference_audio": str(reference_audio_path.relative_to(paths.copied_training_dir)) if reference_audio_path else None,\n        "manifest": str(manifest_path.relative_to(paths.copied_training_dir)) if manifest_path else None,\n        "manifest_summary": manifest or {},\n        "onnx_export": export_report,\n    }\n    paths.metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef upload_package(paths: PackagePaths, repo_id: str, revision: str, hf_folder: str, token: str | None) -> None:\n    from huggingface_hub import HfApi\n\n    if not token:\n        raise RuntimeError("HF_TOKEN ausente. Crie um Kaggle Secret chamado HF_TOKEN para enviar ao Hugging Face.")\n    api = HfApi(token=token)\n    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)\n    LOGGER.info("Enviando pacote para %s/%s", repo_id, hf_folder)\n    api.upload_folder(\n        repo_id=repo_id,\n        repo_type="model",\n        revision=revision,\n        folder_path=str(paths.staging_root),\n        path_in_repo=hf_folder.strip("/"),\n        commit_message=f"Add F5-TTS ONNX package at {hf_folder}",\n    )\n\n\ndef package_voice(args: argparse.Namespace) -> None:\n    LOGGER.info("Voz_Noslen ONNX packager versao: %s", PACKAGER_VERSION)\n    token = get_hf_token()\n    upload_repo_id = args.upload_repo_id or args.repo_id\n    revision = args.revision\n    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")\n    hf_folder = args.hf_folder or f"onnx_packages/voz_noslen_f5tts_onnx_{timestamp}"\n\n    source, source_label = download_source(args.source, args.repo_id, revision, token, args.voice_dir, args.download_mode)\n    paths = make_package_paths()\n    move_tree(source, paths.copied_training_dir)\n\n    manifest_path = find_manifest(paths.copied_training_dir)\n    manifest = load_json_if_exists(manifest_path)\n    checkpoint_path = find_checkpoint(paths.copied_training_dir, manifest, manifest_path)\n    vocab_path = find_vocab(paths.copied_training_dir, checkpoint_path)\n    reference_audio_path = find_reference_audio(paths.copied_training_dir, args.voice_dir)\n\n    LOGGER.info("Checkpoint escolhido: %s", checkpoint_path)\n    LOGGER.info("Vocab escolhido: %s", vocab_path)\n    LOGGER.info("Referencia de audio: %s", reference_audio_path or "nao encontrada")\n\n    export_report: dict[str, Any]\n    try:\n        export_report = export_f5_core_to_onnx(checkpoint_path, vocab_path, paths.onnx_dir, manifest)\n    except Exception as exc:\n        export_report = {\n            "status": "failed",\n            "packager_version": PACKAGER_VERSION,\n            "error_type": type(exc).__name__,\n            "error": str(exc),\n            "traceback": "".join(traceback.format_exception(type(exc), exc, exc.__traceback__)),\n            "note": "A copia dos arquivos originais foi preservada. Corrija a exportacao antes de usar este pacote como ONNX.",\n        }\n        paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding="utf-8")\n        if not args.allow_failed_export:\n            LOGGER.error("Exportacao ONNX falhou. Relatorio: %s", paths.export_report_path)\n            raise\n\n    paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding="utf-8")\n    write_package_metadata(\n        paths,\n        source_label,\n        revision,\n        hf_folder,\n        checkpoint_path,\n        vocab_path,\n        reference_audio_path,\n        manifest_path,\n        manifest,\n        export_report,\n    )\n\n    if args.upload:\n        if not upload_repo_id:\n            raise RuntimeError(\n                "Para fazer upload, informe --upload-repo-id com o Model Repo de destino. "\n                "Buckets nao aceitam upload via HfApi.upload_folder."\n            )\n        upload_package(paths, upload_repo_id, revision, hf_folder, token)\n    else:\n        LOGGER.info("Upload desativado. Pacote local pronto em %s", paths.staging_root)\n\n    LOGGER.info("Pacote final: %s", paths.staging_root)\n    LOGGER.info("Pasta alvo no Hugging Face: %s", hf_folder)\n\n\ndef parse_args(argv: list[str]) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Empacota uma voz F5-TTS em um pacote ONNX no Kaggle.")\n    parser.add_argument("--source", default=os.environ.get("HF_SOURCE_URL", DEFAULT_SOURCE_URL), help="URL ou repo_id do Hugging Face.")\n    parser.add_argument("--repo-id", default=os.environ.get("HF_SOURCE_REPO_ID"), help="Repo ID explicito, ex: warllem/Voz_Noslen.")\n    parser.add_argument("--upload-repo-id", default=os.environ.get("HF_UPLOAD_REPO_ID"), help="Model Repo onde a pasta nova sera enviada.")\n    parser.add_argument("--voice-dir", default=os.environ.get("HF_VOICE_DIR", DEFAULT_VOICE_DIR), help="Pasta da voz dentro do bucket/repo.")\n    parser.add_argument(\n        "--download-mode",\n        choices=("essential", "all"),\n        default=os.environ.get("HF_DOWNLOAD_MODE", "essential"),\n        help="essential baixa apenas a voz escolhida e evita checkpoints duplicados; all tenta baixar tudo.",\n    )\n    parser.add_argument("--revision", default=os.environ.get("HF_REVISION", DEFAULT_REVISION))\n    parser.add_argument("--hf-folder", default=os.environ.get("HF_TARGET_FOLDER"), help="Nova pasta dentro do repo Hugging Face.")\n    parser.add_argument("--upload", action="store_true", default=os.environ.get("HF_UPLOAD", "1") == "1")\n    parser.add_argument("--no-upload", action="store_false", dest="upload")\n    parser.add_argument(\n        "--allow-failed-export",\n        action="store_true",\n        help="Ainda cria e envia o pacote copiado se a exportacao ONNX falhar. Nao recomendado para pacote final.",\n    )\n    return parser.parse_args(argv)\n\n\ndef main(argv: list[str] | None = None) -> None:\n    args = parse_args(argv or sys.argv[1:])\n    package_voice(args)\n\n\nif __name__ == "__main__":\n    main()\n'

requirements_path = WORK / 'requirements_f5_tts_onnx.txt'
script_path = WORK / 'f5_tts_onnx_packager_kaggle.py'
requirements_path.write_text(requirements_text, encoding='utf-8')
script_path.write_text(script_code, encoding='utf-8')

print('Requirements:', requirements_path)
print('Script:', script_path)
print('Arquivos prontos em /kaggle/working')


In [ ]:
# 2) Instalar dependencias
import subprocess
import sys
from pathlib import Path

requirements_path = Path('/kaggle/working/requirements_f5_tts_onnx.txt')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])


In [ ]:
# 3) Configurar origem bucket, voz escolhida e destino model repo
import os
from datetime import datetime, timezone

HF_SOURCE_URL = 'https://huggingface.co/buckets/warllem/Voz_Noslen'
HF_REVISION = 'main'
HF_VOICE_DIR = 'voices/v_minha_voz_f5_tts_ptbr'
HF_DOWNLOAD_MODE = 'essential'
HF_UPLOAD_REPO_ID = 'warllem/Voz_Noslen_ONNX'
HF_TARGET_FOLDER = 'onnx_packages/voz_noslen_f5tts_onnx_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

os.environ['HF_SOURCE_URL'] = HF_SOURCE_URL
os.environ['HF_REVISION'] = HF_REVISION
os.environ['HF_VOICE_DIR'] = HF_VOICE_DIR
os.environ['HF_DOWNLOAD_MODE'] = HF_DOWNLOAD_MODE
os.environ['HF_UPLOAD_REPO_ID'] = HF_UPLOAD_REPO_ID
os.environ['HF_TARGET_FOLDER'] = HF_TARGET_FOLDER
os.environ['HF_UPLOAD'] = '1'

print('Origem:', HF_SOURCE_URL)
print('Voz:', HF_VOICE_DIR)
print('Modo download:', HF_DOWNLOAD_MODE)
print('Repo destino:', HF_UPLOAD_REPO_ID)
print('Pasta nova no Hugging Face:', HF_TARGET_FOLDER)


In [ ]:
# 4) Verificar token Hugging Face no Kaggle Secret
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN encontrado no Kaggle Secret.')
    else:
        print('HF_TOKEN vazio. O download publico pode funcionar, mas upload vai falhar.')
except Exception as exc:
    print('Nao consegui ler Kaggle Secret HF_TOKEN:', exc)
    print('Se o bucket for privado ou se quiser fazer upload, configure HF_TOKEN.')


In [ ]:
# 5) Rodar empacotamento completo e upload para Hugging Face
import subprocess
import sys
from pathlib import Path

script_path = Path('/kaggle/working/f5_tts_onnx_packager_kaggle.py')
cmd = [
    sys.executable,
    str(script_path),
    '--source', HF_SOURCE_URL,
    '--revision', HF_REVISION,
    '--voice-dir', HF_VOICE_DIR,
    '--download-mode', HF_DOWNLOAD_MODE,
    '--upload-repo-id', HF_UPLOAD_REPO_ID,
    '--hf-folder', HF_TARGET_FOLDER,
]
print('Executando:', ' '.join(cmd))
subprocess.check_call(cmd)


In [ ]:
# 6) Conferir arquivos gerados
from pathlib import Path
import json

package_dir = Path('/kaggle/working/voz_noslen_onnx_package')
print('Pacote local:', package_dir)
for path in sorted(package_dir.rglob('*')):
    if path.is_file():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f'{path.relative_to(package_dir)}  {size_mb:.2f} MB')

metadata_path = package_dir / 'package_metadata.json'
report_path = package_dir / 'onnx_export_report.json'
if metadata_path.exists():
    print('
Metadata:')
    print(json.dumps(json.loads(metadata_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:4000])
if report_path.exists():
    print('
Relatorio ONNX:')
    print(json.dumps(json.loads(report_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:4000])


In [ ]:
# 7) Opcional: rodar sem upload, apenas para teste local
# Descomente e execute esta celula se quiser testar sem enviar nada ao Hugging Face.
# import subprocess, sys
# from pathlib import Path
# script_path = Path('/kaggle/working/f5_tts_onnx_packager_kaggle.py')
# subprocess.check_call([
#     sys.executable,
#     str(script_path),
#     '--source', HF_SOURCE_URL,
#     '--revision', HF_REVISION,
#     '--voice-dir', HF_VOICE_DIR,
#     '--download-mode', HF_DOWNLOAD_MODE,
#     '--hf-folder', HF_TARGET_FOLDER,
#     '--no-upload',
# ])
